# converge_bind on a Colab GPU

Bootstraps the whole pipeline on a free Colab T4: clones the repo, downloads SKEMPI 2.0
directly from BSC, extracts every feature, and runs the GPU sweeps that have never been run
on a GPU.

**Before running:** Runtime -> Change runtime type -> Hardware accelerator -> **T4 GPU**.

Colab disconnects after ~90 min idle and ~12 h total. Every step below is idempotent, and
section 6 checkpoints results to Drive, so a disconnect costs you the current step only.


## 1. Confirm the GPU

If this prints `cuda False`, the runtime type is still CPU. Fix it before continuing.

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

## 2. Clone the repo

In [ ]:
import os
REPO = "https://github.com/Tom-Touati/Converge-Affinity-Prediction.git"
BRANCH = "label-defects-and-augmentation-options"

if not os.path.isdir("/content/converge_bind"):
    !git clone --branch {BRANCH} {REPO} /content/converge_bind
os.chdir("/content/converge_bind")
!git log --oneline -3

## 3. Data

SKEMPI 2.0 is served openly by BSC. The two files are 1.6 MB and 30 MB, so this takes
seconds - no upload, no Drive, no scp. The sizes are asserted against the copies the
reported results were produced with, so a truncated or changed download fails loudly here
rather than silently shifting every number downstream.

In [ ]:
import os, pathlib
pathlib.Path("data").mkdir(exist_ok=True)

FILES = {
    "data/skempi_v2.csv":      ("https://life.bsc.es/pid/skempi2/database/download/skempi_v2.csv",      1602208),
    "data/SKEMPI2_PDBs.tgz":   ("https://life.bsc.es/pid/skempi2/database/download/SKEMPI2_PDBs.tgz", 30482090),
}
for path, (url, expect) in FILES.items():
    if not os.path.exists(path) or os.path.getsize(path) != expect:
        !curl -fL --retry 3 -o {path} {url}
    got = os.path.getsize(path)
    assert got == expect, f"{path}: got {got} bytes, expected {expect}"
    print(f"ok  {path}  {got:,} bytes")

## 4. Environment

Two deliberate deviations from `requirements.txt`:

- **torch is not installed.** The pin `torch==2.4.1` exists only for the Windows dev box
  (MSVC runtime 14.28). Installing it here would replace Colab's CUDA build with a CPU
  wheel - exactly backwards. Colab's preinstalled torch is kept.
- **`transformers` is added.** It is a lazy import inside `src/features/hf_plm.py` and is
  missing from `requirements.txt`; without it the ESM-2 steps fail.

In [ ]:
# every pin except torch, which Colab already provides with CUDA
!grep -v '^torch==' requirements.txt > /tmp/req_colab.txt
!pip install -q -r /tmp/req_colab.txt
!pip install -q 'transformers==4.46.3' antiberty

import torch
print("torch still CUDA:", torch.__version__, torch.cuda.is_available())

## 5. Full extraction + baseline

`setup_remote.sh --big` clones ProteinMPNN, then runs every extraction step, skipping any
whose output already exists. `--big` adds **ESM-2 650M**, which is the reason for the GPU:
35M took 16 minutes on a laptop CPU and 650M has never been run at all.

Expect roughly 30-60 minutes on a T4. It ends by retraining the random forest baseline
(the 0.487-0.498 model to beat) and running the test suite.

Note: the script creates its own `.venv`. On Colab that venv would not see the CUDA torch
Colab preinstalled, so `VENV_ARGS=--system-site-packages` is passed through to `venv`.

In [ ]:
# --system-site-packages lets the venv see Colab's CUDA torch; without it the script
# builds a blind venv and installs a CPU wheel over the top.
!VENV_ARGS=--system-site-packages bash scripts/setup_remote.sh --big 2>&1 | tail -60

## 6. Checkpoint to Drive (recommended)

Colab wipes `/content` on disconnect. This copies the extracted features and all run
reports to Drive so a dropped session does not cost you the extraction.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DEST = "/content/drive/MyDrive/converge_bind_runs"
!mkdir -p {DEST}
!rsync -a data/features/ {DEST}/features/
!rsync -a reports/      {DEST}/reports/
!du -sh {DEST}

## 7. The runs the GPU is for

Three open questions, none of which has been answered on a GPU:

1. **`rank_fusion`** - AbRank-style pairwise logistic ranking loss. A CPU sweep was
   launched and never finished.
2. **`fusion_v3`** - chemistry as a third modality alongside sequence and structure.
   Written, never evaluated.
3. **`compare_models`** - the head-to-head table across everything.

Run them one cell at a time so a disconnect does not lose all three.

In [ ]:
!.venv/bin/python -m src.rank_fusion --sweep --device cuda --max-steps 200 --seeds 5 2>&1 | tail -40

In [ ]:
!.venv/bin/python -m src.fusion_v3 --sweep --seeds 5 2>&1 | tail -40

In [ ]:
!.venv/bin/python -m src.compare_models 2>&1 | tail -40

## 8. Does ESM-2 650M rescue the sequence modality?

The single most informative question this GPU can answer. Six probes have shown the
sequence arm contributes nothing over structure+chemistry, with two diagnosed mechanisms
(PLM likelihood is inverted for binding; the embedding difference is 82% substitution
identity). Both are arguments about *what the model represents*, not about capacity - so
if 650M also fails, that is a real finding, not a compute limit.

Compare against the 0.487 baseline from section 5.

In [ ]:
!.venv/bin/python -m src.train --model rf --features chem,geom,geomrev,mpnn,esm650M_pair     --name colab_rf_esm650M --n-boot 1000 2>&1 | tail -25

## 9. Pull everything back

Re-run the Drive sync, then download a zip of just the reports.

In [ ]:
!rsync -a reports/ {DEST}/reports/
!zip -qr /content/reports.zip reports
from google.colab import files
files.download('/content/reports.zip')